
# Data Visualization

## Project: Delivery Time Deviation Prediction in Logistics

This notebook visualizes logistics delivery data to support the project objective: improving ETA reliability by analyzing delivery time deviation patterns.

## Required Visualizations

The original visualization requirements are:

1. Scatter plot: delivery time vs distance.
2. Boxplot: duration by hour/day.
3. Heatmap: late delivery rate by zone-hour.
4. Line chart: average duration trend.

Because the selected dataset does not contain direct columns such as `distance`, `duration`, `vehicle_type`, or `delivery_zone`, proxy variables are created from available logistics variables.

## Proxy Variable Mapping

| Original Requirement | Direct Column Available? | Proxy Used |
|---|---:|---|
| Distance | No | `distance_proxy` from `eta_variation_hours` |
| Delivery duration | No | `final_delivery_time_hours` from logistics time-related columns |
| Delivery zone | No | `gps_zone` from GPS latitude and longitude bins |
| Late delivery | No direct flag | `is_late` from `delivery_time_deviation > 0` |

These proxy variables allow the visualization phase to follow the original analytical goals while remaining consistent with the actual dataset.



## 1. Setup and Load Dataset


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)

cleaned_file = Path("cleaned_dynamic_supply_chain_logistics_dataset.csv")
original_file = Path("dynamic_supply_chain_logistics_dataset.csv")

if cleaned_file.exists():
    df = pd.read_csv(cleaned_file)
    print("Loaded cleaned dataset.")
elif original_file.exists():
    df = pd.read_csv(original_file)
    print("Loaded original dataset.")
else:
    raise FileNotFoundError("Dataset file not found. Please place the CSV file in the same folder as this notebook.")

print("Dataset shape:", df.shape)
display(df.head())



## 2. Prepare Features for Visualization

This step creates the variables required for visualization:
- `hour_of_day`
- `day_of_week`
- `month`
- `is_late`
- `distance_proxy`
- `final_delivery_time_hours`
- `gps_zone`


In [ ]:

# Convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Time-based features
df["hour_of_day"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()
df["month"] = df["timestamp"].dt.to_period("M").astype(str)

# Late delivery flag
# If delivery_time_deviation > 0, the delivery is later than expected
df["is_late"] = (df["delivery_time_deviation"] > 0).astype(int)

# Distance proxy
# Actual distance is not available, so ETA variation is used as a proxy
df["distance_proxy"] = df["eta_variation_hours"]

# Delivery duration proxy
# Actual delivery duration is not available, so a proxy is created from time-related logistics columns
df["final_delivery_time_hours"] = (
    df["lead_time_days"] * 24
    + df["loading_unloading_time"]
    + df["customs_clearance_time"]
    + df["eta_variation_hours"]
)

# Remove invalid duration proxy values if any
df = df[df["final_delivery_time_hours"] > 0].reset_index(drop=True)

# GPS-based delivery zone
df["lat_bin"] = pd.cut(df["vehicle_gps_latitude"], bins=5, labels=False)
df["long_bin"] = pd.cut(df["vehicle_gps_longitude"], bins=5, labels=False)
df["gps_zone"] = df["lat_bin"].astype(str) + "_" + df["long_bin"].astype(str)

display(df[[
    "timestamp",
    "hour_of_day",
    "day_of_week",
    "month",
    "distance_proxy",
    "final_delivery_time_hours",
    "gps_zone",
    "delivery_time_deviation",
    "is_late"
]].head())



## 3. Scatter Plot: Delivery Time vs Distance Proxy

### Purpose

The original requirement is to create a scatter plot of delivery time vs distance. Since the dataset does not contain actual distance, `eta_variation_hours` is used as `distance_proxy`. The delivery time is represented by `final_delivery_time_hours`.

This plot helps analyze whether longer estimated delivery variation is associated with longer delivery time.


In [ ]:

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="distance_proxy",
    y="final_delivery_time_hours",
    alpha=0.4
)

plt.title("Scatter Plot: Delivery Time vs Distance Proxy")
plt.xlabel("Distance Proxy (ETA Variation Hours)")
plt.ylabel("Final Delivery Time Proxy (Hours)")
plt.grid(True, alpha=0.3)
plt.show()



### Interpretation

Each point represents one logistics record. The x-axis shows the distance proxy and the y-axis shows the delivery time proxy. If the points show an upward trend, it suggests that larger ETA variation is associated with longer delivery time.



## 4. Scatter Plot: Traffic Congestion vs Delivery Time Deviation

### Purpose

This plot directly supports the project objective by showing the relationship between traffic congestion and delivery time deviation.


In [ ]:

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="traffic_congestion_level",
    y="delivery_time_deviation",
    alpha=0.4
)

plt.title("Scatter Plot: Traffic Congestion vs Delivery Time Deviation")
plt.xlabel("Traffic Congestion Level")
plt.ylabel("Delivery Time Deviation")
plt.grid(True, alpha=0.3)
plt.show()



### Interpretation

This scatter plot helps determine whether higher traffic congestion is associated with larger delivery time deviation. If higher congestion values correspond to higher deviation values, traffic is likely an important factor affecting ETA reliability.



## 5. Boxplot: Duration by Hour

### Purpose

The original requirement is to create a boxplot of duration by hour. Since actual duration is not available, this notebook uses `delivery_time_deviation` and `final_delivery_time_hours` as delivery performance measures.

This first boxplot shows how delivery time deviation varies by hour of day.


In [ ]:

plt.figure(figsize=(14, 6))

sns.boxplot(
    data=df,
    x="hour_of_day",
    y="delivery_time_deviation"
)

plt.title("Boxplot: Delivery Time Deviation by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Delivery Time Deviation")
plt.grid(True, axis="y", alpha=0.3)
plt.show()



### Interpretation

This boxplot compares delivery time deviation across different hours. It can reveal whether certain hours have higher deviation or more unstable delivery performance, which may indicate peak-hour effects.



## 6. Boxplot: Duration by Day of Week

### Purpose

This plot shows how delivery time deviation changes across weekdays and weekends.


In [ ]:

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="day_of_week",
    y="delivery_time_deviation",
    order=day_order
)

plt.title("Boxplot: Delivery Time Deviation by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Delivery Time Deviation")
plt.xticks(rotation=45)
plt.grid(True, axis="y", alpha=0.3)
plt.show()



### Interpretation

This boxplot helps identify whether delivery deviation differs between weekdays and weekends. It can support scheduling and operational planning if specific days show higher delivery uncertainty.



## 7. Heatmap: Late Delivery Rate by Zone and Hour

### Purpose

The original requirement is to create a heatmap of late delivery rate by zone-hour. Since the dataset does not contain a direct delivery zone column, `gps_zone` is created from latitude and longitude bins.

Late delivery is defined as:

`delivery_time_deviation > 0`


In [ ]:

zone_hour_late_rate = df.groupby(["gps_zone", "hour_of_day"])["is_late"].mean().reset_index()
zone_hour_late_rate["late_rate_percent"] = zone_hour_late_rate["is_late"] * 100

heatmap_data = zone_hour_late_rate.pivot(
    index="gps_zone",
    columns="hour_of_day",
    values="late_rate_percent"
)

plt.figure(figsize=(15, 8))

sns.heatmap(
    heatmap_data,
    cmap="Reds",
    annot=False
)

plt.title("Heatmap: Late Delivery Rate by GPS Zone and Hour")
plt.xlabel("Hour of Day")
plt.ylabel("GPS-Based Zone")
plt.show()



### Interpretation

Darker cells represent higher late delivery rates. This heatmap helps identify which GPS-based zones and hours are more likely to experience late deliveries. It is useful for delivery managers who want to improve scheduling or allocate more resources to risky zone-hour combinations.



## 8. Line Chart: Average Duration Trend

### Purpose

The original requirement is to create a line chart of average duration trend. Since actual duration is not available, this notebook uses the average delivery time deviation over months.


In [ ]:

monthly_trend = df.groupby("month")["delivery_time_deviation"].mean().reset_index()

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=monthly_trend,
    x="month",
    y="delivery_time_deviation",
    marker="o"
)

plt.title("Line Chart: Average Delivery Time Deviation Trend by Month")
plt.xlabel("Month")
plt.ylabel("Average Delivery Time Deviation")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.show()



### Interpretation

This line chart shows the trend of average delivery time deviation over time. It helps identify whether delivery deviation increases, decreases, or fluctuates across months.



## 9. Bar Chart: Average Delivery Deviation by Risk Classification

### Purpose

This additional chart is useful because the dataset contains `risk_classification`. It helps compare delivery deviation across risk groups.


In [ ]:

risk_avg = df.groupby("risk_classification")["delivery_time_deviation"].mean().reset_index()

plt.figure(figsize=(8, 5))

sns.barplot(
    data=risk_avg,
    x="risk_classification",
    y="delivery_time_deviation"
)

plt.title("Average Delivery Time Deviation by Risk Classification")
plt.xlabel("Risk Classification")
plt.ylabel("Average Delivery Time Deviation")
plt.grid(True, axis="y", alpha=0.3)
plt.show()



### Interpretation

This bar chart shows whether higher risk categories are associated with larger delivery time deviation. If high-risk records have higher average deviation, risk classification can be useful for ETA prediction and delivery monitoring.



## 10. Correlation Heatmap of Numerical Features

### Purpose

A correlation heatmap helps identify numerical variables that may be related to delivery time deviation.


In [ ]:

numeric_df = df.select_dtypes(include=["int64", "float64"])

corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 10))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap of Numerical Features")
plt.show()



### Interpretation

This heatmap shows correlations between numerical variables. Strong positive or negative correlations with `delivery_time_deviation` may indicate important factors for machine learning modeling.



## 11. Focused Correlation Heatmap: Top Features Related to Delivery Time Deviation

### Purpose

The full correlation heatmap can be hard to read. This chart focuses on the features most related to `delivery_time_deviation`.


In [ ]:

target_col = "delivery_time_deviation"

target_corr = corr_matrix[target_col].abs().sort_values(ascending=False)

top_features = target_corr.head(10).index.tolist()

plt.figure(figsize=(10, 8))

sns.heatmap(
    df[top_features].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    center=0
)

plt.title("Correlation Heatmap of Top Features Related to Delivery Time Deviation")
plt.show()

display(target_corr.head(10))



### Interpretation

This heatmap highlights the variables most strongly related to delivery time deviation. These features may be useful for later regression modeling and feature selection.



## 12. Summary of Data Visualization

This notebook creates all required visualizations adapted to the available dataset:

1. Scatter plot of delivery time vs distance proxy.
2. Boxplot of delivery time deviation by hour.
3. Boxplot of delivery time deviation by day of week.
4. Heatmap of late delivery rate by GPS-based zone and hour.
5. Line chart of average delivery time deviation trend.
6. Additional bar chart by risk classification.
7. Correlation heatmaps for numerical feature relationships.

These plots support the project by identifying patterns related to ETA deviation, traffic, time, zone, and risk classification.
